# Demo 6 — Build a three-tier router (and then question the number)

**AI Cost Management and Token Utilization** · Module 3 · ~15 minutes

> Runs top-to-bottom on live API keys. Every cell that spends money prints what it spent.

---

## What this demo lands

1. A cascade with a calibrated confidence signal cuts blended cost substantially.
2. **Cost per token is the wrong unit.** Divide by success rate — and add cleanup cost.
3. Benchmark headlines (85–98%) and production reality (UCCI: **31%**, 95% CI 27–35%) are different numbers.
   Budget for the second and celebrate the first.

In [ ]:
# --- Setup: install + keys -------------------------------------------------
# Colab: this cell installs everything. Local: it is a no-op if already installed.
%pip install -q anthropic openai tiktoken pandas matplotlib 2>/dev/null

import os, getpass

def need(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
    return os.environ[var]

# You need at least one. Anthropic is used for the cache-metadata demos because
# it reports cache reads in a separate, easily inspectable bucket.
need('ANTHROPIC_API_KEY')
# need('OPENAI_API_KEY')   # uncomment if you want the OpenAI comparisons
print('keys loaded')

In [ ]:
# --- Verified rate card, September 2026 ------------------------------------
# Sources (checked 5 Sept 2026):
#   platform.claude.com/docs/en/about-claude/pricing
#   developers.openai.com/api/docs/pricing
#   ai.google.dev/gemini-api/docs/pricing
#   deepseek.ai/pricing
# USD per 1,000,000 tokens.  cache_w = 5-minute cache write, cache_r = cache read.

PRICES = {
    # model id                       input  output  cache_w  cache_r
    'deepseek-v4-flash':            dict(inp=0.14, out=0.28,  cw=0.14,  cr=0.0028),
    'gpt-5.6-luna':                 dict(inp=0.20, out=1.20,  cw=0.25,  cr=0.02),
    'gemini-3.5-flash-lite':        dict(inp=0.30, out=2.50,  cw=0.30,  cr=0.03),
    'gemini-3.8-flash':             dict(inp=0.75, out=3.75,  cw=0.75,  cr=0.075),
    'claude-haiku-4-5':             dict(inp=1.00, out=5.00,  cw=1.25,  cr=0.10),
    'claude-sonnet-5':              dict(inp=2.00, out=10.00, cw=2.50,  cr=0.20),
    'gpt-5.6-terra':                dict(inp=2.00, out=12.00, cw=2.50,  cr=0.20),
    'claude-opus-5':                dict(inp=5.00, out=25.00, cw=6.25,  cr=0.50),
    'gpt-5.6-sol':                  dict(inp=4.00, out=20.00, cw=5.00,  cr=0.40),
    'gpt-6-astra':                  dict(inp=10.00,out=50.00, cw=12.50, cr=1.00),
    'claude-fable-5-1':             dict(inp=10.00,out=50.00, cw=12.50, cr=0.25),
}

def cost(model, inp=0, out=0, cache_w=0, cache_r=0):
    """Cost in USD for one call, given token counts by billing bucket."""
    p = PRICES[model]
    return (inp*p['inp'] + out*p['out'] + cache_w*p['cw'] + cache_r*p['cr']) / 1e6

def usd(x):
    return f'${x:,.6f}' if x < 0.01 else f'${x:,.4f}' if x < 1 else f'${x:,.2f}'

print(f'{len(PRICES)} models loaded')

In [ ]:
# --- Cost ledger: every billable call in this notebook lands here ----------
import pandas as pd
LEDGER = []

def log_call(label, model, inp=0, out=0, cache_w=0, cache_r=0, note=''):
    c = cost(model, inp, out, cache_w, cache_r)
    LEDGER.append(dict(label=label, model=model, input=inp, output=out,
                       cache_write=cache_w, cache_read=cache_r, usd=c, note=note))
    print(f'{label:<38} {usd(c):>12}   in={inp:<7} out={out:<6} cw={cache_w:<7} cr={cache_r:<7} {note}')
    return c

def ledger():
    df = pd.DataFrame(LEDGER)
    if df.empty:
        print('no calls yet'); return df
    print(f'\nTOTAL SPENT IN THIS NOTEBOOK: {usd(df.usd.sum())}')
    return df

In [ ]:
import anthropic, re, json
import pandas as pd
client = anthropic.Anthropic()

TIERS = {
  'floor':        'claude-haiku-4-5',
  'near_frontier':'claude-sonnet-5',
  'frontier':     'claude-opus-5',
}
# Map to model ids your key can actually call:
MODEL_IDS = {
  'claude-haiku-4-5':  'claude-haiku-4-5',
  'claude-sonnet-5':   'claude-sonnet-5',
  'claude-opus-5':     'claude-opus-5',
}
print(TIERS)

---
## 1. A mixed workload — 70% simple, 20% medium, 10% hard

This distribution is what production telemetry consistently shows. It is why routing works.

In [ ]:
TASKS = (
  # simple: extraction / classification — floor tier work
  [dict(q=f'Extract the order id from: "Order NW-{1000+i} was delayed 3 days." '
          f'Reply with the id only.', gold=f'NW-{1000+i}', klass='simple') for i in range(14)]
  # medium: grounded reasoning over a short policy
+ [dict(q='Policy: refunds under $500 auto-approve if delay > 48h and fewer than 3 prior claims. '
        f'Customer: ${300+i*40} claim, 60h delay, {i%4} prior claims. Auto-approve? '
        'Answer YES or NO first.',
        gold='YES' if (300+i*40) < 500 and (i%4) < 3 else 'NO', klass='medium') for i in range(4)]
  # hard: multi-constraint synthesis
+ [dict(q='Three shipments: A delayed 50h claim $480 with 2 prior claims; B delayed 20h claim '
        '$100 with 0 prior; C delayed 72h claim $900 with 1 prior. Policy: auto-approve if '
        'delay>48h AND claim<$500 AND prior<3; else escalate. Which auto-approve? '
        'List the letters only.', gold='A', klass='hard') for _ in range(2)]
)
print(f'{len(TASKS)} tasks: ',
      pd.Series([t["klass"] for t in TASKS]).value_counts().to_dict())

---
## 2. Baseline — everything on the near-frontier tier (what most teams actually do)

In [ ]:
def call(model, prompt, max_tokens=120):
    r = client.messages.create(model=MODEL_IDS[model], max_tokens=max_tokens,
                               messages=[{'role':'user','content':prompt}])
    return r.content[0].text.strip(), r.usage

def graded(answer, gold):
    return gold.lower() in answer.lower()

baseline = []
for t in TASKS:
    ans, u = call('claude-sonnet-5', t['q'])
    ok = graded(ans, t['gold'])
    c = cost('claude-sonnet-5', u.input_tokens, u.output_tokens)
    baseline.append(dict(klass=t['klass'], model='claude-sonnet-5', cost=c, ok=ok))

bdf = pd.DataFrame(baseline)
B_COST, B_ACC = bdf.cost.sum(), bdf.ok.mean()
print(f'BASELINE  cost={usd(B_COST)}  accuracy={B_ACC:.0%}  ({len(TASKS)} tasks, all near-frontier)')

---
## 3. The cascade — cheap first, escalate on low confidence

The confidence signal matters more than the routing logic. UCCI (arXiv:2605.18796) showed
isotonic calibration of token-level uncertainty beat entropy thresholding, conformal prediction,
and FrugalGPT-style routing at identical accuracy — improving calibration error from 0.12 to 0.03.

In [ ]:
def confidence_probe(model, prompt, answer):
    """Cheap self-verification: ask the SAME cheap model whether its answer is well-supported.
    Crude but real — and it is what a production cascade does before paying to escalate."""
    probe = (f'Question: {prompt}\n\nProposed answer: {answer}\n\n'
             'Is this answer fully determined by the question, with no ambiguity? '
             'Reply with only HIGH, MEDIUM or LOW.')
    verdict, u = call(model, probe, max_tokens=8)
    score = {'HIGH': 0.95, 'MEDIUM': 0.6, 'LOW': 0.2}.get(verdict.strip().upper()[:6], 0.5)
    return score, cost(model, u.input_tokens, u.output_tokens)

THRESHOLD = 0.9   # tune this on a validation set — do not guess it in production

routed = []
for t in TASKS:
    spend, path = 0.0, []

    # tier 1: floor
    ans, u = call('claude-haiku-4-5', t['q'])
    spend += cost('claude-haiku-4-5', u.input_tokens, u.output_tokens); path.append('floor')
    conf, probe_cost = confidence_probe('claude-haiku-4-5', t['q'], ans)
    spend += probe_cost

    if conf < THRESHOLD:
        # tier 2: near-frontier
        ans, u = call('claude-sonnet-5', t['q'])
        spend += cost('claude-sonnet-5', u.input_tokens, u.output_tokens); path.append('near_frontier')
        conf, probe_cost = confidence_probe('claude-haiku-4-5', t['q'], ans)
        spend += probe_cost
        if conf < 0.7:
            # tier 3: frontier
            ans, u = call('claude-opus-5', t['q'])
            spend += cost('claude-opus-5', u.input_tokens, u.output_tokens); path.append('frontier')

    ok = graded(ans, t['gold'])
    routed.append(dict(klass=t['klass'], path='>'.join(path), tiers=len(path), cost=spend, ok=ok))
    print(f"{t['klass']:<8} {'>'.join(path):<32} {usd(spend):>11}  {'OK' if ok else 'MISS'}")

rdf = pd.DataFrame(routed)
R_COST, R_ACC = rdf.cost.sum(), rdf.ok.mean()

---
## 4. The result — and the reframe

In [ ]:
print('ROUTING MIX')
display(rdf.groupby('path').agg(n=('cost','size'), cost=('cost','sum'), acc=('ok','mean')))

print(f'\n{"":<22}{"cost":>12}{"accuracy":>11}{"cost/task":>12}')
print(f'{"baseline (all mid)":<22}{usd(B_COST):>12}{B_ACC:>11.0%}{usd(B_COST/len(TASKS)):>12}')
print(f'{"3-tier cascade":<22}{usd(R_COST):>12}{R_ACC:>11.0%}{usd(R_COST/len(TASKS)):>12}')
print(f'\nRaw cost saving: {1-R_COST/B_COST:>.0%}')

### Now divide by the success rate

$$E[\text{cost per solved task}] = \frac{C_{attempt}}{p_{success}} + L \times K_{cleanup}$$

Where `L` is the leak rate of wrong outputs reaching a business process and `K_cleanup`
is what a human costs to unwind one. **Cleanup frequently exceeds the API bill.**

In [ ]:
CLEANUP_COST = 12.00   # USD of human time to unwind one wrong answer. Set this honestly.

def per_solved(total_cost, accuracy, n, cleanup=CLEANUP_COST):
    attempt = total_cost / n
    leaked  = 1 - accuracy
    return attempt / max(accuracy, 1e-9) + leaked * cleanup

b = per_solved(B_COST, B_ACC, len(TASKS))
r = per_solved(R_COST, R_ACC, len(TASKS))

print(f'{"":<22}{"cost/CALL":>13}{"cost/SOLVED TASK":>20}')
print(f'{"baseline":<22}{usd(B_COST/len(TASKS)):>13}{usd(b):>20}')
print(f'{"cascade":<22}{usd(R_COST/len(TASKS)):>13}{usd(r):>20}')
print()
if r < b:
    print(f'The cascade wins on BOTH units. Saving per solved task: {1-r/b:.0%}')
else:
    print('The cascade is cheaper per call and MORE EXPENSIVE per solved task.')
    print('This is the trap. Reliability is a cost lever. Raise the escalation threshold.')
print()
print('Try setting CLEANUP_COST to 0, then to 50. Watch the correct decision flip.')

---
## 5. Sensitivity — where does the decision actually flip?

In [ ]:
rows = []
for cu in [0, 1, 5, 12, 25, 50, 100]:
    rows.append(dict(cleanup=cu,
                     baseline=per_solved(B_COST, B_ACC, len(TASKS), cu),
                     cascade=per_solved(R_COST, R_ACC, len(TASKS), cu)))
sdf = pd.DataFrame(rows)
sdf['cascade_wins'] = sdf.cascade < sdf.baseline
display(sdf.style.format({'baseline':'${:,.4f}','cascade':'${:,.4f}'}))
print('\nThis table, not the raw cost saving, is what you take to the decision meeting.')

In [ ]:
ledger()

---
## Takeaways

- Ship the **quality gate in the same pull request** as the router. A router without an eval
  harness is an unmonitored quality regression.
- Route on **task class, risk, latency budget and data sensitivity** — never input length alone.
- Calibrate the confidence signal on a validation set. UCCI's whole contribution is that
  raw confidence scores are miscalibrated and threshold-picking should be an optimisation, not a guess.
- Expect **~31% on a real workload**, not the 85–98% you see in benchmark papers with wide model spreads.